# Plot saved Fourier symmetrization results

Run this notebook after `01_train_and_save_data.ipynb`.  It reads local `.npz` files from `data/` and writes only `qpinn_boundary_constraint_loss_comparison_side_by_side.png` and `qpinn_boundary_constraint_solution_comparison_combined.png` to `figures/`.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile

import numpy as np

_MPL_CACHE = Path(tempfile.gettempdir()) / "matplotlib-cache"
_MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPL_CACHE))
_XDG_CACHE = Path(tempfile.gettempdir()) / "xdg-cache"
_XDG_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(_XDG_CACHE))

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    markers = (
        "01_train_and_save_data.ipynb",
        "02_plot_saved_data.ipynb",
        "requirements.txt",
    )
    for candidate in [cwd, *cwd.parents]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    return cwd


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "data"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading data from {DATA_DIR}")
print(f"Writing figures to {FIGURE_DIR}")


In [ ]:
QPINN_DISPLAY_LABELS = {
    "Ordinary QFM": "Fourier-QPINN",
    "Cheb-QFM": "Chebyshev-QPINN",
    "$B_2$-QCM": "$B_2$-QPINN",
}
QPINN_MODEL_SLUGS = {
    "Ordinary QFM": "ordinary_qfm",
    "Cheb-QFM": "cheb_qfm",
    "$B_2$-QCM": "b2_qcm",
}
PINN_DISPLAY_LABEL = "Fully connected PINN"
PINN_MODEL_SLUG = "fc_pinn"
PINN_CONSTRAINT_SLUGS = {"Hard constraint": "hard", "Soft constraint": "soft"}
PINN_LABELS = ("Hard constraint", "Soft constraint")
PINN_COLORS = {
    "Fourier-QPINN": "#1f77b4",
    "Chebyshev-QPINN": "#7b2cbf",
    "$B_2$-QPINN": "#d62728",
    PINN_DISPLAY_LABEL: "#00A6A6",
}



def pinn_locate_comparison_data_paths() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    candidates = []
    if "DATA_DIR" in globals():
        candidates.append(Path(DATA_DIR))
    candidates.extend([cwd / "data", cwd])
    for parent in [cwd, *cwd.parents]:
        candidates.extend([parent / "data", parent])

    seen = set()
    for directory in candidates:
        directory = directory.resolve()
        if directory in seen:
            continue
        seen.add(directory)
        qpinn_path = directory / "qpinn_boundary_constraint_comparison_data.npz"
        pinn_path = directory / "pinn_boundary_constraint_comparison_data.npz"
        if qpinn_path.exists() and pinn_path.exists():
            return qpinn_path, pinn_path
    raise FileNotFoundError("Could not find both QPINN and PINN comparison .npz files")
def pinn_show_or_close_current_figure() -> None:
    if "show_or_close_current_figure" in globals():
        show_or_close_current_figure()
    elif "ipykernel" in sys.modules:
        plt.show()
    else:
        plt.close()


qpinn_data_path, pinn_data_path = pinn_locate_comparison_data_paths()
qpinn_data = np.load(qpinn_data_path)
pinn_data = np.load(pinn_data_path)
PINN_FIGURE_DIR = Path(globals().get("FIGURE_DIR", qpinn_data_path.parent.parent / "figures"))
PINN_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


LOSS_YLIM = (1.0e-5, 1.0e2)
LOSS_YTICKS = [1.0e-5, 1.0e-4, 1.0e-3, 1.0e-2, 1.0e-1, 1.0, 1.0e1, 1.0e2]
LOSS_DRAW_ORDER = [
    (PINN_DISPLAY_LABEL, None),
    ("Fourier-QPINN", "Ordinary QFM"),
    ("Chebyshev-QPINN", "Cheb-QFM"),
    ("$B_2$-QPINN", "$B_2$-QCM"),
]
LOSS_LEGEND_ORDER = ["Fourier-QPINN", "Chebyshev-QPINN", "$B_2$-QPINN", PINN_DISPLAY_LABEL]
LOSS_ZORDER = {
    PINN_DISPLAY_LABEL: 2,
    "Fourier-QPINN": 3,
    "Chebyshev-QPINN": 4,
    "$B_2$-QPINN": 6,
}


def plot_loss_lines(ax, constraint_slug: str, metric: str) -> dict[str, object]:
    handles_by_label = {}
    for display_label, qpinn_name in LOSS_DRAW_ORDER:
        if display_label == PINN_DISPLAY_LABEL:
            data = pinn_data
            prefix = f"{constraint_slug}_{PINN_MODEL_SLUG}"
            linewidth = 2.8
        else:
            data = qpinn_data
            prefix = f"{constraint_slug}_{QPINN_MODEL_SLUGS[qpinn_name]}"
            linewidth = 3.0 if display_label == "$B_2$-QPINN" else 2.6

        mean = data[f"{prefix}_{metric}_geo_mean"]
        lower = data[f"{prefix}_{metric}_geo_lower"]
        upper = data[f"{prefix}_{metric}_geo_upper"]
        steps = np.arange(1, len(mean) + 1)
        zorder = LOSS_ZORDER[display_label]
        ax.fill_between(
            steps,
            lower,
            upper,
            color=PINN_COLORS[display_label],
            alpha=0.14,
            linewidth=0,
            zorder=zorder - 0.5,
        )
        (line,) = ax.plot(
            steps,
            mean,
            label=display_label,
            color=PINN_COLORS[display_label],
            linewidth=linewidth,
            zorder=zorder,
        )
        handles_by_label[display_label] = line
    return handles_by_label


def style_loss_axis(ax, title: str, ylabel: str) -> None:
    ax.set_yscale("log")
    ax.set_ylim(*LOSS_YLIM)
    ax.set_yticks(LOSS_YTICKS)
    ax.set_yticklabels([r"$10^{-5}$", r"$10^{-4}$", r"$10^{-3}$", r"$10^{-2}$", r"$10^{-1}$", r"$10^{0}$", r"$10^{1}$", r"$10^{2}$"])
    ax.set_title(title, fontsize=26, pad=12)
    ax.set_xlabel("Training step", fontsize=24, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=24, labelpad=10)
    ax.tick_params(axis="both", which="major", labelsize=18, width=1.1, length=5)
    ax.tick_params(axis="both", which="minor", width=0.9, length=3)
    ax.grid(True, which="major", alpha=0.25)
    ax.grid(True, which="minor", alpha=0.08)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)


def plot_qpinn_pinn_loss_pair() -> Path:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7.8), dpi=220, sharex=True, sharey=True)
    configs = {
        "Hard constraint": {"metric": "residual", "ylabel": "Total loss"},
        "Soft constraint": {"metric": "objective", "ylabel": ""},
    }

    legend_handles = None
    for ax, constraint_label in zip(axes, PINN_LABELS):
        constraint_slug = PINN_CONSTRAINT_SLUGS[constraint_label]
        metric = configs[constraint_label]["metric"]
        handles_by_label = plot_loss_lines(ax, constraint_slug, metric)
        if legend_handles is None:
            legend_handles = [handles_by_label[label] for label in LOSS_LEGEND_ORDER]
        style_loss_axis(ax, constraint_label, configs[constraint_label]["ylabel"])

    fig.legend(
        legend_handles,
        LOSS_LEGEND_ORDER,
        frameon=False,
        loc="lower center",
        ncol=4,
        fontsize=21,
        handlelength=2.4,
    )
    fig.tight_layout(rect=(0.0, 0.13, 1.0, 1.0))
    path = PINN_FIGURE_DIR / "qpinn_boundary_constraint_loss_comparison_side_by_side.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    pinn_show_or_close_current_figure()
    return path



side_by_side_loss_path = plot_qpinn_pinn_loss_pair()
print(f"saved side-by-side hard/soft loss comparison: {side_by_side_loss_path}")

q_grid_x1 = qpinn_data["grid_x1"]
q_grid_x2 = qpinn_data["grid_x2"]
q_target_grid = qpinn_data["target_grid"]
pinn_predictions = {}
pinn_errors = {}
for constraint_label in PINN_LABELS:
    constraint_slug = PINN_CONSTRAINT_SLUGS[constraint_label]
    prefix = f"{constraint_slug}_{PINN_MODEL_SLUG}"
    pinn_predictions[constraint_label] = pinn_data[f"{prefix}_prediction_grid"]
    pinn_errors[constraint_label] = float(pinn_data[f"{prefix}_error"])

qpin_predictions = {}
qpin_errors = {}
for constraint_label in PINN_LABELS:
    constraint_slug = PINN_CONSTRAINT_SLUGS[constraint_label]
    for qpinn_name in QPINN_DISPLAY_LABELS:
        model_slug = QPINN_MODEL_SLUGS[qpinn_name]
        prefix = f"{constraint_slug}_{model_slug}"
        key = (constraint_label, qpinn_name)
        qpin_predictions[key] = qpinn_data[f"{prefix}_prediction_grid"]
        qpin_errors[key] = float(qpinn_data[f"{prefix}_error"])

combined_values = [q_target_grid, *qpin_predictions.values(), *pinn_predictions.values()]
combined_vmin = min(float(np.min(values)) for values in combined_values)
combined_vmax = max(float(np.max(values)) for values in combined_values)
combined_columns = ["Analytical", *QPINN_DISPLAY_LABELS.keys(), PINN_DISPLAY_LABEL]

fig, axes = plt.subplots(
    len(PINN_LABELS),
    len(combined_columns),
    figsize=(23.0, 8.0),
    constrained_layout=True,
    squeeze=False,
)
assert axes.size == 10
last_image = None
for row, constraint_label in enumerate(PINN_LABELS):
    for col, column_name in enumerate(combined_columns):
        ax = axes[row, col]
        if column_name == "Analytical":
            values = q_target_grid
            title = "Analytical"
        elif column_name == PINN_DISPLAY_LABEL:
            values = pinn_predictions[constraint_label]
            title = f"{PINN_DISPLAY_LABEL}\nerror={pinn_errors[constraint_label]:.2e}"
        else:
            key = (constraint_label, column_name)
            values = qpin_predictions[key]
            display_label = QPINN_DISPLAY_LABELS[column_name]
            title = f"{display_label}\nerror={qpin_errors[key]:.2e}"

        last_image = ax.contourf(
            q_grid_x1,
            q_grid_x2,
            values,
            levels=56,
            cmap="Blues",
            vmin=combined_vmin,
            vmax=combined_vmax,
        )
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=18, pad=9)
        ax.set_xticks([-1, 0, 1])
        ax.set_yticks([-1, 0, 1])
        ax.tick_params(axis="both", labelsize=13, width=1.0, length=4)
        if row == len(PINN_LABELS) - 1:
            ax.set_xlabel("$x_1$", fontsize=16, labelpad=2)
        if col == 0:
            ax.set_ylabel(f"{constraint_label}\n$x_2$", fontsize=18, labelpad=7)
        for spine in ax.spines.values():
            spine.set_linewidth(1.0)

# fig.suptitle("Hard vs soft constraint", fontsize=24)
colorbar = fig.colorbar(last_image, ax=axes.ravel().tolist(), shrink=0.82, pad=0.012)
colorbar.set_label("$u(x_1,x_2)$", fontsize=18, labelpad=8)
colorbar.ax.tick_params(labelsize=14, width=1.0, length=4)
combined_path = PINN_FIGURE_DIR / "qpinn_boundary_constraint_solution_comparison_combined.png"
fig.savefig(combined_path, dpi=240, bbox_inches="tight")
pinn_show_or_close_current_figure()
print(f"saved combined QPINN/PINN solution comparison: {combined_path}")
